<!--
Copyright (c) 2026 F. Hoffmann-La Roche Ltd
Authors:
- ERP Technology Platform Team <glospot_members@msxdl.roche.com>
-->

# Check the PSEs of STRUST

Every PSE (Personal Security Environment) listed in transaction `STRUST` should
exist in the database and match its copy in the file system of the application
server. The function module `SSFR_PSE_CHECK` (function group `SSFR`) does that
check for one PSE and is remote-enabled, so it can be called over RFC.

This notebook runs it on system **EH4** for one PSE and then for all of them:

1. `SSFR_GET_ALL_STRUST_IDENTITIES` lists the PSEs shown in `STRUST`.
2. `SSFR_PSE_CHECK` checks a PSE. Its parameters are:

   | Direction | Name | Meaning |
   |-----------|------|---------|
   | importing | `IS_STRUST_IDENTITY` | The PSE to check: structure with `PSE_CONTEXT` and `PSE_APPLIC` (and `SPRSL` and `PSE_DESCRIPT`, which the check does not use). |
   | exporting | `EV_PSE_EXISTING` | `X` if the PSE is stored in the database, blank if not. |
   | exporting | `EV_CRC` | Result code of the check of the PSE file on the application server: `0` is OK. |
   | tables | `ET_BAPIRET2` | Messages describing the outcome. |

It does so with the two clients of `dragonrfc`:

- `dragonrfc.RFCClient`: the synchronous client, which blocks until the call returns.
- `dragonrfc.AsyncRFCClient`: the asyncio client, which runs the same call on a
  worker thread so the event loop stays free.

The logon uses **SNC with Kerberos**, so no password appears anywhere.

## Prerequisites

1. The SAP NW RFC SDK is installed and `dragonrfc` is built (`task setup`).
2. MIT Kerberos 5 is installed. Its GSSAPI library is the SNC library:
   `libgssapi_krb5.so.2` on Linux, `libgssapi_krb5.dylib` on macOS and
   `gssapi64.dll` on Windows.
3. You hold a valid Kerberos ticket for your user. Get one with
   `kinit prietoj4@EMEA.ROCHE.COM` and check it with `klist`.
4. Your user in EH4, client 010, is mapped to the SNC name
   `p:prietoj4@EMEA.ROCHE.COM` (transaction `SU01`, tab *SNC*) and has the
   authorizations to call `SSFR_GET_ALL_STRUST_IDENTITIES` and `SSFR_PSE_CHECK`
   over RFC (`S_RFC`). The check reads PSEs, so a missing authorization surfaces
   as an error message in the result (`SSFPSE_CHECK` raises `AUTHORITY_MISSING`).

## Step 1: Logon parameters

`RFCClient` takes the SDK logon parameters as keyword arguments, all of them
`str`. All examples below share the same dictionary.

**Load-balanced logon through the message server**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `mshost` | `eh4scs.sap.roche.com` | Host of the message server (ASCS instance). |
| `msserv` | `3600` | Message server port (`36<instance number>`). |
| `sysid` | `EH4` | System ID; the message server serves only this system. |
| `group` | `SPACE` | Logon group (`SMLG`); the message server picks the least loaded application server in it. |
| `client` | `010` | SAP client. |

**SNC instead of user and password**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `snc_mode` | `1` | Turns SNC on. The SAP user is derived from the SNC name, so `user` and `passwd` are not set. |
| `snc_qop` | `9` | Quality of protection: `9` asks for the maximum the SNC library offers (integrity and privacy, i.e. encryption). |
| `snc_myname` | `p:prietoj4@EMEA.ROCHE.COM` | Your own SNC name: the Kerberos principal of your ticket. |
| `snc_partnername` | `p:sapEH4/EH4@EMEA.ROCHE.COM` | SNC name of the SAP system: the Kerberos service principal of EH4. |
| `snc_lib` | GSSAPI library | Path or name of the MIT Kerberos GSSAPI library. |

The cell finds the GSSAPI library with `ctypes.util.find_library`, which works on
every OS when MIT Kerberos is installed in the default location. If it does not
find it (for example, Homebrew's keg-only `krb5` on macOS), set the `SNC_LIB`
environment variable to the full path of the library before starting Jupyter.

In [2]:
import asyncio
import ctypes.util
import os

import dragonrfc

SNC_LIB = os.environ.get("SNC_LIB") or ctypes.util.find_library(
    "gssapi64" if os.name == "nt" else "gssapi_krb5",
)
assert SNC_LIB, "MIT Kerberos GSSAPI library not found: set SNC_LIB"

LOGON = {
    # Load balancing through the message server
    "mshost": "eh4scs.sap.roche.com",
    "msserv": "3600",
    "sysid": "EH4",
    "group": "SPACE",
    "client": "010",
    # SNC with Kerberos (GSSAPI)
    "snc_mode": "1",
    "snc_qop": "9",
    "snc_myname": "p:prietoj4@EMEA.ROCHE.COM",
    "snc_partnername": "p:sapEH4/EH4@EMEA.ROCHE.COM",
    "snc_lib": SNC_LIB,
}

print(f"dragonrfc {dragonrfc.__version__}, SDK {dragonrfc.sdk_version()}")
print(f"SNC library: {SNC_LIB}")

ModuleNotFoundError: No module named 'dragonrfc._core'

## Step 2: Check connectivity

Before calling any function module, make sure EH4 is reachable and accepts the
SNC logon. Opening the connection already performs the whole logon (message
server lookup, SNC handshake with your Kerberos ticket, user check);
`ping` then sends a round trip over the open connection.

Failures surface as typed exceptions, which the cell reports instead of a
traceback:

- `dragonrfc.RFCCommunicationError`: the message server or the application
  server cannot be reached (wrong host or port, firewall, VPN off), or the SNC
  library failed (for example, no valid Kerberos ticket).
- `dragonrfc.RFCLogonError`: EH4 rejected the logon (SNC name not mapped to a
  user, user locked, wrong client).

The same check is shown with both clients. Expected output: `EH4 reachable (sync)`
and `EH4 reachable (async)`. Continue only if both succeed.

In [ ]:
try:
    with dragonrfc.RFCClient(**LOGON) as conn:
        conn.ping()
except (dragonrfc.RFCCommunicationError, dragonrfc.RFCLogonError) as error:
    print(f"EH4 not reachable (sync): {type(error).__name__}: {error.message}")
else:
    print("EH4 reachable (sync)")

try:
    async with await dragonrfc.AsyncRFCClient.open(**LOGON) as aconn:
        await aconn.ping()
except (dragonrfc.RFCCommunicationError, dragonrfc.RFCLogonError) as error:
    print(f"EH4 not reachable (async): {type(error).__name__}: {error.message}")
else:
    print("EH4 reachable (async)")

## Step 3: List the PSEs of `STRUST` (synchronous)

`SSFR_GET_ALL_STRUST_IDENTITIES` returns the PSEs that `STRUST` shows in its
tree. It has no importing parameters. In the result:

- `ET_STRUST_IDENTITIES` holds one row per PSE, with the fields `PSE_CONTEXT`,
  `PSE_APPLIC`, `SPRSL` (language) and `PSE_DESCRIPT` (description). Context and
  application together identify a PSE. A row has exactly the layout of the
  `IS_STRUST_IDENTITY` parameter of `SSFR_PSE_CHECK`, so it can be passed on
  as is.
- `ET_BAPIRET2` holds messages. The function module reports errors there
  instead of raising exceptions, so the cell checks it for messages of type `E`
  (error) or `A` (abort).

`call` blocks the current thread until EH4 answers; it releases the GIL while
waiting, so other Python threads keep running. The `with` block closes the
connection at the end, even if an error is raised.

Expected output: one line per PSE with its context, application and description.

In [ ]:
with dragonrfc.RFCClient(**LOGON) as conn:
    result = conn.call("SSFR_GET_ALL_STRUST_IDENTITIES")

errors = [msg["MESSAGE"] for msg in result["ET_BAPIRET2"] if msg["TYPE"] in {"E", "A"}]
assert not errors, errors

identities = result["ET_STRUST_IDENTITIES"]
for identity in identities:
    context, applic = identity["PSE_CONTEXT"], identity["PSE_APPLIC"]
    print(f"{context:<4} {applic:<8} {identity['PSE_DESCRIPT']}")

## Step 4: Check one PSE (synchronous)

The cell checks the System PSE, context `PROG` and application `<SYST>`. A
structure parameter is passed as a `dict` keyed by field name; fields left out
stay blank, so `PSE_CONTEXT` and `PSE_APPLIC` are enough.

The function module first derives the file name of the PSE (the same rule as
`SSFPSE_FILENAME`), looks for it in the database, and then checks the copy on
the application server your connection landed on. Every outcome is reported in
`ET_BAPIRET2`, not as an exception:

- `EV_PSE_EXISTING = 'X'`, `EV_CRC = 0` and one message of type `S` (`Local PSE
  OK`): the PSE is fine.
- `EV_PSE_EXISTING` blank and a message of type `E` (`PSE missing on
  database`): the PSE is listed in `STRUST` but was never created.
- `EV_CRC` not `0` and a message of type `E`: the PSE file on the application
  server is missing or does not match the database copy. The message number is
  `026 + EV_CRC`, in message class `TRUST`.

`show` prints the outcome of one PSE as a line; it is a plain function, so the
steps below reuse it.

Expected output: `PROG <SYST>  existing=X crc=0 S: Local PSE OK (SSFPSE_CHECK:PROG/<SYST>)`
on a healthy system.

In [ ]:
def show(identity: dict[str, str], result: dict[str, object]) -> None:
    """Prints the outcome of SSFR_PSE_CHECK for one PSE."""
    messages = "; ".join(f"{m['TYPE']}: {m['MESSAGE']}" for m in result["ET_BAPIRET2"])
    existing = result["EV_PSE_EXISTING"] or "-"
    print(
        f"{identity['PSE_CONTEXT']:<4} {identity['PSE_APPLIC']:<8} "
        f"existing={existing} crc={result['EV_CRC']} {messages}",
    )


system_pse = {"PSE_CONTEXT": "PROG", "PSE_APPLIC": "<SYST>"}
with dragonrfc.RFCClient(**LOGON) as conn:
    result = conn.call("SSFR_PSE_CHECK", IS_STRUST_IDENTITY=system_pse)
show(system_pse, result)

## Step 5: Check all PSEs (synchronous)

The cell checks every PSE from step 3 on one connection, one call after the
other. A connection runs one call at a time, so this is the simplest way and
needs a single logon.

Keep in mind that the check of the PSE file runs on the application server
that holds your connection. The logon group `SPACE` can hand out a different
server for every new connection, so a PSE that is missing on one server only
may show up as fine or as an error depending on where you land. Set `ashost`
and `sysnr` in `LOGON` instead of the message server parameters to pin the
check to one server.

Expected output: one line per PSE, as in step 4.

In [ ]:
with dragonrfc.RFCClient(**LOGON) as conn:
    for identity in identities:
        show(identity, conn.call("SSFR_PSE_CHECK", IS_STRUST_IDENTITY=identity))

## Step 6: The same, asynchronously and concurrently

`AsyncRFCClient` has the same methods as `RFCClient`, but they are coroutines.
`AsyncRFCClient.open` opens the connection on a worker thread, so the logon
does not block the event loop either; `async with` closes it at the end.

The checks do not depend on each other, so they can run at the same time. One
connection runs one call at a time (the SDK forbids sharing a connection between
threads), so `check` opens its own connection per PSE and `asyncio.gather`
runs them concurrently. The price is one logon per PSE, so for a long list
limit the number of connections, for example with `asyncio.Semaphore`. If a
call fails, `gather` raises its exception and the cell stops.

Jupyter runs an event loop already, so `await` works at the top level of a
cell. In a script, put the code in an `async def main()` function and run it
with `asyncio.run(main())`.

Expected output: the same lines as in step 5, in the same order (`gather`
returns the results in the order of its arguments).

In [ ]:
async def check(identity: dict[str, str]) -> dict[str, object]:
    """Runs SSFR_PSE_CHECK for one PSE on its own connection."""
    async with await dragonrfc.AsyncRFCClient.open(**LOGON) as aconn:
        return await aconn.call("SSFR_PSE_CHECK", IS_STRUST_IDENTITY=identity)


results = await asyncio.gather(*(check(identity) for identity in identities))
for identity, result in zip(identities, results, strict=True):
    show(identity, result)

## Summary

- `SSFR_PSE_CHECK` checks one `STRUST` PSE per call. It reports problems in
  `ET_BAPIRET2` and `EV_CRC` instead of raising exceptions, so inspect the
  result: `EV_PSE_EXISTING` blank or `EV_CRC` not `0` means the PSE needs
  attention in `STRUST`.
- `SSFR_GET_ALL_STRUST_IDENTITIES` supplies the PSEs to check, and its rows are
  valid input for `SSFR_PSE_CHECK`.
- The check of the PSE file runs on the application server of the connection;
  pin the server with `ashost` and `sysnr` if the result must be reproducible.
- Both clients take the same logon parameters and return the same results and
  errors; only the calling style differs.
- Use `RFCClient` in scripts, command-line tools and threaded code: it is the
  simplest choice and a `with` block guarantees the connection is closed.
- Use `AsyncRFCClient` inside asyncio applications (web services, bots,
  notebooks running many checks): calls do not block the event loop, and
  separate connections run concurrently.
- SNC with Kerberos keeps credentials out of the code: the SAP user comes from
  your Kerberos ticket and QoP `9` encrypts the traffic.